## Build Agents with LangChain
Build ReAct (Reasoning + Action) agents

### Setup

In [1]:
!pip install langchain langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.1 MB/s eta 0:00:00


In [2]:
# Set Groq API token
# In Prod should be done through .env files
import os
os.environ['GROQ_API_KEY'] = "<>"

###The Brain (Model)

In [3]:
# Model - The Brain
from langchain_groq import ChatGroq

model = ChatGroq(
    model = "llama-3.1-8b-instant",
    max_tokens = 1000
)


In [4]:
# Invoke to get the response
res = model.invoke("Explain quantum physics in one line")

In [6]:
res.content

'Quantum physics is the study of the behavior of matter and energy at the smallest scales, where particles can exist in multiple states simultaneously and their properties are probabilistic rather than definite.'

### Type of Messages

In [13]:
# Types of Messages in LangChain
# {"role":"user", "content": "Explain in one line"}
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

conversation = [
    SystemMessage("You are helpful assistant"),
    HumanMessage("Explain quantum physics"),
    {"role":"user", "content": "Explain in one line"}
]

res = model.invoke(conversation)
res

AIMessage(content='Quantum physics is a branch of physics that studies the behavior of matter and energy at an atomic and subatomic level, where particles can exist in multiple states and locations simultaneously, governed by the principles of wave-particle duality and uncertainty.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 53, 'total_tokens': 102, 'completion_time': 0.062172543, 'completion_tokens_details': None, 'prompt_time': 0.006238893, 'prompt_tokens_details': None, 'queue_time': 0.134583427, 'total_time': 0.068411436}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_020e283281', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fd1cd-417c-7e93-bbec-a563963c9c88-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 53, 'output_tokens': 49, 'total_tokens': 102})

###Tools (the Hands)

In [14]:
# Get Weather tool. Langchain registers it as Structured Tool
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
  """Returns temperature of a city(input) in degree celsius(string)"""
  #api call
  res = 20
  return str(res)

In [22]:
print(get_weather.args)

{'city': {'title': 'City', 'type': 'string'}}


In [23]:
# Suggest Activity Tool
@tool
def suggest_activity(temperature: int) -> str:
  """Suggests activity to do as per Temperature as input"""

  # Logic

  return "Jump on Bed"

In [25]:
print(suggest_activity.args)

{'temperature': {'title': 'Temperature', 'type': 'integer'}}


In [ ]:
# All tools


###Create Agent (Model + Tools + ReAct Loop)

In [53]:
# Create Agent
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model = model,
    tools = [get_weather, suggest_activity],
    system_prompt = SystemMessage("You are an Activity Suggester based on weather"),
    checkpointer = InMemorySaver()
)

In [54]:
# Result using invoke
config = {"configurable": {"thread_id" : "chat_id_1"}}

res_agent = agent.invoke(
    {"messages": [HumanMessage("Based on waether in Mysore suggest me an activity")]}, config = config
)

In [55]:
for msg in res_agent['messages']:
  print(msg.type, "->", msg.content)

human -> Based on waether in Mysore suggest me an activity
ai -> 
tool -> 20
ai -> 
tool -> Jump on Bed
ai -> It seems there was an error in the temperature provided, as 20 is not a typical temperature.


In [47]:
# Result using Stream
for step in agent.stream({"messages": [HumanMessage("Based on waether in Mysore suggest me an activity")]}):
  print(step)

{'model': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'rf4m6x24r', 'function': {'arguments': '{"city":"Mysore"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 307, 'total_tokens': 337, 'completion_time': 0.099330235, 'completion_tokens_details': None, 'prompt_time': 0.071129118, 'prompt_tokens_details': None, 'queue_time': 0.049296513, 'total_time': 0.170459353}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_020e283281', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fd201-289b-70d0-8da8-50e77bd0e61e-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Mysore'}, 'id': 'rf4m6x24r', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 307, 'output_tokens': 30, 'total_tokens': 337})]}}
{'tools': {'messages': [ToolMessage(content='20', name='get_weather', id='9

###Save in Memory (Persistance)

In [56]:
# First invoke with in-memory saver
config = {"configurable": {"thread_id" : "chat_id_1"}}

res = agent.invoke({"messages": [HumanMessage("What was the name of the city?")]}, config = config)

In [60]:
for msg in res['messages']:
  print(msg.type, msg.content)

human Based on waether in Mysore suggest me an activity
ai 
tool 20
ai 
tool Jump on Bed
ai It seems there was an error in the temperature provided, as 20 is not a typical temperature.
human What was the name of the city?
ai Mysore


In [57]:
res['messages'][-1].content

'Mysore'

In [ ]:
# Follow up Question


###Subagents (Wrap a LLM as Tool)

In [64]:
# Create Research Agent (Sub Agent)

#model
from langchain_groq import ChatGroq

model_r = ChatGroq(
    model = "llama-3.3-70b-versatile",
    max_tokens = 1000
)

# agent
from langchain.agents import create_agent

agent_r = create_agent(
    model = model_r,
    system_prompt = SystemMessage("You are a Research Agent")
)

# tool
@tool
def research_agent(query: str) -> str:
  """This is a research Agent who works on the query"""
  res = agent_r.invoke({"messages": [HumanMessage(query)]})

  return res['messages'][-1].content

In [65]:
# Create Main Agent (Supervisor)
from langchain_groq import ChatGroq

model_s = ChatGroq(
    model = "llama-3.1-8b-instant",
    max_tokens = 1000
)

# agent
from langchain.agents import create_agent

agent_s = create_agent(
    model = model_s,
    tools = [research_agent],
    system_prompt = SystemMessage("You are the Supervisor Agent")
)



In [66]:
# Stream output
for step in agent_s.stream({"messages": [HumanMessage("Research and Explain Quantum Physics in 1 line")]}):
  print(step)

{'model': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'by7q0cnz3', 'function': {'arguments': '{"query":"Explain Quantum Physics in 1 line"}', 'name': 'research_agent'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 234, 'total_tokens': 257, 'completion_time': 0.03988093, 'completion_tokens_details': None, 'prompt_time': 0.0127938, 'prompt_tokens_details': None, 'queue_time': 0.170637447, 'total_time': 0.05267473}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_03e8423237', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fd216-e2d8-76c0-9f34-bdb724c9e8e5-0', tool_calls=[{'name': 'research_agent', 'args': {'query': 'Explain Quantum Physics in 1 line'}, 'id': 'by7q0cnz3', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 234, 'output_tokens': 23, 'total_tokens': 257})]}}
{'tools': {'messa